# 🔬 Diabetic Retinopathy Classification — EfficientNet-B3/B4 Ensemble
## IDRiD Dataset | Transfer Learning | Kaggle T4 x2 GPU

**Paper**: Porwal et al., *"Indian Diabetic Retinopathy Image Dataset (IDRiD)"*, MDPI Data, 2018

**Task**: 5-class DR severity grading (Grade 0–4) using EfficientNet ensemble with transfer learning

**Hardware**: Kaggle T4 x2 (2× NVIDIA Tesla T4, 16GB VRAM each)

---

## 📦 Section 1: Environment Setup & Imports

In [1]:
# ============================================================
# Install dependencies
# ============================================================
!pip install -q timm albumentations scikit-learn matplotlib seaborn
print("✅ All dependencies installed!")

✅ All dependencies installed!


In [2]:
# ============================================================
# Imports
# ============================================================
import os
import gc
import copy
import time
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm import tqdm
print(1)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LambdaLR

import timm
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, cohen_kappa_score,
    f1_score, accuracy_score
)

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns

warnings.filterwarnings('ignore')
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"timm version: {timm.__version__}")

1
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
timm version: 1.0.25


## ⚙️ Section 2: Configuration

In [3]:
# ============================================================
# Centralized Configuration
# ============================================================
CONFIG = {
    # ─── Model ───────────────────────────────────────────────
    "model_names": ["efficientnet_b3", "efficientnet_b4"],
    "num_classes": 5,
    "pretrained": True,
    "drop_rate": 0.3,

    # ─── Training ────────────────────────────────────────────
    "img_size": 512,
    "batch_size": 8,        
    "batch_size_b4": 8,      # B4 needs lower batch size to avoid GPU OOM
    "num_epochs": 30,
    "lr": 1e-4,
    "weight_decay": 1e-5,
    "warmup_epochs": 3,      # freeze backbone epochs
    "label_smoothing": 0.1,
    "mixup_alpha": 0.2,

    # ─── Augmentation ────────────────────────────────────────
    "use_clahe": True,
    "clahe_clip_limit": 2.0,
    "clahe_grid_size": 8,

    # ─── Validation ──────────────────────────────────────────
    "val_ratio": 0.2,
    "num_workers": 0,
    "seed": 42,

    # ─── Checkpointing ───────────────────────────────────────
    "output_dir": "/kaggle/working/",
    "monitor_metric": "val_qwk",

    # ─── Class Names ─────────────────────────────────────────
    "class_names": [
        "Grade 0 - No DR",
        "Grade 1 - Mild NPDR",
        "Grade 2 - Moderate NPDR",
        "Grade 3 - Severe NPDR",
        "Grade 4 - Proliferative DR",
    ],
}

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)
for name in CONFIG["model_names"]:
    os.makedirs(os.path.join(CONFIG["output_dir"], name), exist_ok=True)
os.makedirs(os.path.join(CONFIG["output_dir"], "ensemble"), exist_ok=True)

print("✅ Configuration loaded")
print(f"   Models: {CONFIG['model_names']}")
print(f"   Image size: {CONFIG['img_size']}×{CONFIG['img_size']}")
print(f"   Batch size (default): {CONFIG['batch_size']}")
print(f"   Batch size (B4): {CONFIG['batch_size_b4']}")
print(f"   Epochs: {CONFIG['num_epochs']}")
print(f"   Learning rate: {CONFIG['lr']}")

✅ Configuration loaded
   Models: ['efficientnet_b3', 'efficientnet_b4']
   Image size: 512×512
   Batch size (default): 8
   Batch size (B4): 8
   Epochs: 30
   Learning rate: 0.0001


## 🔧 Section 3: Utility Functions

In [4]:
# ============================================================
# Seed & Device
# ============================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Seed set to {CONFIG['seed']}")
print(f"✅ Device: {DEVICE}")

✅ Seed set to 42
✅ Device: cuda


In [5]:
# ============================================================
# Auto-detect dataset paths (Kaggle or local)
# ============================================================
def find_dataset_paths():
    # Kaggle paths
    kaggle_base = Path("/kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset")
    # Local paths (Windows)
    local_base = Path(r"a:/VHU/Dataset")

    for base in [kaggle_base, local_base]:
        grading = base / "B.%20Disease%20Grading" / "B. Disease Grading"
        if not grading.exists():
            grading = base / "B. Disease Grading"
        if grading.exists():
            train_img = grading / "1. Original Images" / "a. Training Set"
            test_img = grading / "1. Original Images" / "b. Testing Set"
            gt_dir = grading / "2. Groundtruths"

            train_csv = None
            test_csv = None
            for f in gt_dir.iterdir():
                fname = f.name.lower()
                if "training" in fname and fname.endswith(".csv"):
                    train_csv = f
                elif "testing" in fname and fname.endswith(".csv"):
                    test_csv = f

            if train_img.exists() and train_csv:
                return {
                    "train_img_dir": str(train_img),
                    "test_img_dir": str(test_img),
                    "train_csv": str(train_csv),
                    "test_csv": str(test_csv),
                }
    raise FileNotFoundError("IDRiD dataset not found!")

PATHS = find_dataset_paths()
for k, v in PATHS.items():
    print(f"  {k}: {v}")
print("✅ Dataset paths detected!")

  train_img_dir: /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/1. Original Images/a. Training Set
  test_img_dir: /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/1. Original Images/b. Testing Set
  train_csv: /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
  test_csv: /kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/B.%20Disease%20Grading/B. Disease Grading/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv
✅ Dataset paths detected!


## 📊 Section 4: Data Exploration

In [6]:
# ============================================================
# Load CSV labels and explore class distribution
# ============================================================
train_df = pd.read_csv(PATHS["train_csv"])
test_df = pd.read_csv(PATHS["test_csv"])

# Clean columns - keep only relevant ones
train_df = train_df[["Image name", "Retinopathy grade"]].copy()
test_df = test_df[["Image name", "Retinopathy grade"]].copy()
train_df.columns = ["image_id", "label"]
test_df.columns = ["image_id", "label"]

# Add full image paths
train_df["image_path"] = train_df["image_id"].apply(
    lambda x: os.path.join(PATHS["train_img_dir"], f"{x}.jpg")
)
test_df["image_path"] = test_df["image_id"].apply(
    lambda x: os.path.join(PATHS["test_img_dir"], f"{x}.jpg")
)

# Verify all images exist
train_exists = train_df["image_path"].apply(os.path.exists).sum()
test_exists = test_df["image_path"].apply(os.path.exists).sum()
print(f"Train images found: {train_exists}/{len(train_df)}")
print(f"Test images found:  {test_exists}/{len(test_df)}")

# Class distribution
print("\n📊 Training Set Distribution:")
print(train_df["label"].value_counts().sort_index())
print(f"\n📊 Testing Set Distribution:")
print(test_df["label"].value_counts().sort_index())

Train images found: 413/413
Test images found:  103/103

📊 Training Set Distribution:
label
0    134
1     20
2    136
3     74
4     49
Name: count, dtype: int64

📊 Testing Set Distribution:
label
0    34
1     5
2    32
3    19
4    13
Name: count, dtype: int64


In [7]:
# ============================================================
# Visualize class distribution
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2ecc71', '#f39c12', '#e74c3c', '#9b59b6', '#e91e63']

for ax, df, title in zip(axes, [train_df, test_df], ["Training Set (413)", "Testing Set (103)"]):
    counts = df["label"].value_counts().sort_index()
    bars = ax.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_xlabel("DR Grade", fontsize=12, fontweight='bold')
    ax.set_ylabel("Count", fontsize=12, fontweight='bold')
    ax.set_title(f"Class Distribution — {title}", fontsize=13, fontweight='bold')
    ax.set_xticks(range(5))
    ax.set_xticklabels([f"G{i}" for i in range(5)])
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(count), ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "class_distribution.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Class distribution plotted")

✅ Class distribution plotted


In [8]:
# ============================================================
# Sample images visualization
# ============================================================
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for grade in range(5):
    sample = train_df[train_df["label"] == grade].iloc[0]
    img = cv2.imread(sample["image_path"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (512, 512))
    axes[grade].imshow(img)
    axes[grade].set_title(CONFIG["class_names"][grade], fontsize=11, fontweight='bold')
    axes[grade].axis('off')
plt.suptitle("Sample Fundus Images per DR Grade", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "sample_images.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Sample images displayed")

✅ Sample images displayed


## 🖼️ Section 5: Dataset & Augmentation

In [9]:
# ============================================================
# CLAHE Preprocessing
# ============================================================
def apply_clahe(image, clip_limit=2.0, grid_size=8):
    """Apply CLAHE on L-channel of LAB color space."""
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l_channel = lab[:, :, 0]
    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=(grid_size, grid_size)
    )
    lab[:, :, 0] = clahe.apply(l_channel)
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

# ============================================================
# Augmentation Pipelines
# ============================================================
def get_transforms(phase, img_size=512):
    if phase == "train":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1, scale_limit=0.15,
                rotate_limit=45, p=0.5, border_mode=cv2.BORDER_CONSTANT
            ),
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
                A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=1.0),
            ], p=0.5),
            A.OneOf([
                A.GaussNoise(var_limit=(10, 50), p=1.0),
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            ], p=0.3),
            A.CoarseDropout(
                max_holes=8, max_height=32, max_width=32,
                min_holes=1, min_height=8, min_width=8,
                fill_value=0, p=0.3
            ),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

print("✅ Augmentation pipelines defined")

✅ Augmentation pipelines defined


In [10]:
# ============================================================
# IDRiD Dataset Class
# ============================================================
class IDRiDDataset(Dataset):
    def __init__(self, df, transform=None, use_clahe=True,
                 clahe_clip=2.0, clahe_grid=8, img_size=512):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.use_clahe = use_clahe
        self.img_size = img_size
        self.clahe_clip = clahe_clip
        self.clahe_grid = clahe_grid

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row["image_path"])
        if image is None:
            raise FileNotFoundError(f"Failed to read image: {row['image_path']}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # W1 fix: Resize BEFORE CLAHE (50x faster on 512x512 vs 4288x2848)
        image = cv2.resize(image, (self.img_size, self.img_size),
                          interpolation=cv2.INTER_AREA)

        # Apply CLAHE securely inside worker to prevent OpenCV multiple-worker deadlock
        if self.use_clahe:
            clahe = cv2.createCLAHE(clipLimit=self.clahe_clip, tileGridSize=(self.clahe_grid, self.clahe_grid))
            lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
            lab[:, :, 0] = clahe.apply(lab[:, :, 0])
            image = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

        # Apply augmentations
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        label = torch.tensor(row["label"], dtype=torch.long)
        return image, label

print("✅ IDRiDDataset class defined")

✅ IDRiDDataset class defined


## 📦 Section 6: DataLoader Setup

In [ ]:
# ============================================================
# Stratified Train/Val Split + Weighted Sampler
# ============================================================
# StratifiedShuffleSplit already imported in imports cell

def create_dataloaders(train_df, test_df, config):
    # --- Stratified split ---
    sss = StratifiedShuffleSplit(n_splits=1, test_size=config["val_ratio"],
                                 random_state=config["seed"])
    train_idx, val_idx = next(sss.split(train_df, train_df["label"]))

    df_train = train_df.iloc[train_idx].reset_index(drop=True)
    df_val = train_df.iloc[val_idx].reset_index(drop=True)

    print(f"  Train: {len(df_train)} images")
    print(f"  Val:   {len(df_val)} images")
    print(f"  Test:  {len(test_df)} images")
    print(f"  Train label dist: {dict(Counter(df_train['label']))}")
    print(f"  Val   label dist: {dict(Counter(df_val['label']))}")

    # --- Transforms ---
    train_transform = get_transforms("train", config["img_size"])
    val_transform = get_transforms("val", config["img_size"])

    # --- Datasets ---
    train_dataset = IDRiDDataset(
        df_train, transform=train_transform,
        use_clahe=config["use_clahe"],
        clahe_clip=config["clahe_clip_limit"],
        clahe_grid=config["clahe_grid_size"],
        img_size=config["img_size"],
    )
    val_dataset = IDRiDDataset(
        df_val, transform=val_transform,
        use_clahe=config["use_clahe"],
        clahe_clip=config["clahe_clip_limit"],
        clahe_grid=config["clahe_grid_size"],
        img_size=config["img_size"],
    )
    test_dataset = IDRiDDataset(
        test_df, transform=val_transform,
        use_clahe=config["use_clahe"],
        clahe_clip=config["clahe_clip_limit"],
        clahe_grid=config["clahe_grid_size"],
        img_size=config["img_size"],
    )

    # --- Class weights for weighted sampling ---
    class_counts = Counter(df_train["label"])
    total = len(df_train)
    class_weights_map = {c: total / (len(class_counts) * count)
                         for c, count in class_counts.items()}
    sample_weights = [class_weights_map[label] for label in df_train["label"]]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights),
                                     replacement=True)

    # --- Class weights tensor for loss (used only for reference, NOT in loss) ---
    class_weights = torch.tensor(
        [class_weights_map[i] for i in range(config["num_classes"])],
        dtype=torch.float32
    )

    # --- DataLoaders ---
    train_loader = DataLoader(
        train_dataset, batch_size=config["batch_size"],
        sampler=sampler, num_workers=config["num_workers"],
        pin_memory=False, drop_last=True, persistent_workers=False
    )
    val_loader = DataLoader(
        val_dataset, batch_size=config["batch_size"],
        shuffle=False, num_workers=config["num_workers"],
        pin_memory=False, persistent_workers=False
    )
    test_loader = DataLoader(
        test_dataset, batch_size=config["batch_size"],
        shuffle=False, num_workers=config["num_workers"],
        pin_memory=False, persistent_workers=False
    )

    return train_loader, val_loader, test_loader, class_weights, df_train, df_val

train_loader, val_loader, test_loader, class_weights, df_train_split, df_val_split = \
    create_dataloaders(train_df, test_df, CONFIG)

print(f"\n  Class weights: {class_weights.numpy().round(2)}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")
print("✅ DataLoaders created!")

  Train: 330 images
  Val:   83 images
  Test:  103 images
  Train label dist: {1: 16, 2: 109, 0: 107, 4: 39, 3: 59}
  Val   label dist: {2: 27, 0: 27, 3: 15, 4: 10, 1: 4}

  Class weights: [0.62 4.12 0.61 1.12 1.69]
  Train batches: 41
  Val batches:   11
  Test batches:  13
✅ DataLoaders created!


## 🧠 Section 7: Model Architecture

In [12]:
# ============================================================
# DR Classifier with EfficientNet Backbone
# ============================================================
class DRClassifier(nn.Module):
    """EfficientNet-based DR severity classifier."""
    def __init__(self, model_name, num_classes=5, pretrained=True, drop_rate=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0,  # remove classifier head
            drop_rate=drop_rate,
        )
        in_features = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(drop_rate),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(drop_rate * 0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False
        print("  🔒 Backbone frozen")

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True
        print("  🔓 Backbone unfrozen")

# Test model creation
test_model = DRClassifier("efficientnet_b3", num_classes=5, pretrained=False)
dummy = torch.randn(2, 3, 512, 512)
out = test_model(dummy)
print(f"  Model output shape: {out.shape}")
del test_model, dummy, out
gc.collect()
print("✅ DRClassifier defined and tested")

  Model output shape: torch.Size([2, 5])
✅ DRClassifier defined and tested


## 🏋️ Section 8: Training & Validation Functions

In [13]:
# ============================================================
# Mixup data augmentation
# ============================================================
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ============================================================
# Train one epoch
# ============================================================
def train_one_epoch(model, loader, criterion, optimizer, device, epoch,
                    use_mixup=True, mixup_alpha=0.2, scaler=None):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    pbar = tqdm(loader, desc=f"Epoch {epoch+1} Train", leave=False, dynamic_ncols=True)
    for batch_idx, (images, labels) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # W3 fix: zero_grad BEFORE forward pass
        optimizer.zero_grad()

        # I3: Mixed precision training (AMP)
        with torch.amp.autocast('cuda', enabled=scaler is not None):
            if use_mixup and np.random.random() < 0.5:
                images, labels_a, labels_b, lam = mixup_data(images, labels, mixup_alpha)
                outputs = model(images)
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                # C3 fix: With Mixup, accuracy is approximate (tracked against labels_a only)
                all_labels.extend(labels_a.cpu().numpy())
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc

# ============================================================
# Validate
# ============================================================
@torch.no_grad()
def validate(model, loader, criterion, device, use_amp=False):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    pbar = tqdm(loader, desc="Validation", leave=False, dynamic_ncols=True)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)

        probs = torch.softmax(outputs.float(), dim=1)
        _, preds = torch.max(outputs, 1)
        running_loss += loss.item() * images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')

    return epoch_loss, epoch_acc, epoch_qwk, epoch_f1, all_preds, all_labels, np.array(all_probs)

print("✅ Training & validation functions defined")

✅ Training & validation functions defined


## 💾 Section 9: Checkpointing

In [14]:
# ============================================================
# Save & Load Checkpoints
# ============================================================
def save_checkpoint(model, optimizer, scheduler, epoch, metrics, filepath, config):
    state = {
        "epoch": epoch,
        "model_state_dict": model.module.state_dict() if hasattr(model, 'module') else model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "metrics": metrics,
        "config": config,
    }
    torch.save(state, filepath)

def load_checkpoint(filepath, model, optimizer=None, scheduler=None):
    checkpoint = torch.load(filepath, map_location="cpu")
    if hasattr(model, 'module'):
        model.module.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint["model_state_dict"])
    if optimizer and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler and checkpoint.get("scheduler_state_dict"):
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    return checkpoint.get("epoch", 0), checkpoint.get("metrics", {})

print("✅ Checkpoint functions defined")

✅ Checkpoint functions defined


## 🚀 Section 10 & 11: Training Loop

In [15]:
# ============================================================
# Full Training Pipeline for one model
# ============================================================
def train_model(model_name, train_loader, val_loader, class_weights, config, device):
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"{'='*60}")

    # Create model
    model = DRClassifier(
        model_name=model_name,
        num_classes=config["num_classes"],
        pretrained=config["pretrained"],
        drop_rate=config["drop_rate"],
    )

    # Multi-GPU
    if torch.cuda.device_count() > 1:
        print(f"  Using {torch.cuda.device_count()} GPUs with DataParallel")
        model = nn.DataParallel(model)
    model = model.to(device)

    # Loss — W2 fix: no class weights in loss (WeightedRandomSampler handles imbalance)
    criterion = nn.CrossEntropyLoss(
        label_smoothing=config["label_smoothing"],
    )

    # I3: Mixed Precision scaler for T4 GPUs (~2x speedup)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None
    use_amp = scaler is not None

    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    # Scheduler
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)

    # Training history
    history = {
        "epoch": [], "train_loss": [], "train_acc": [],
        "val_loss": [], "val_acc": [], "val_qwk": [], "val_f1": [], "lr": [],
    }

    best_qwk = -1.0
    patience_counter = 0
    model_dir = os.path.join(config["output_dir"], model_name)
    start_time = time.time()

    start_epoch = 0
    ckpt_path = os.path.join(model_dir, "last_model.pth")
    if os.path.exists(ckpt_path):
        print(f"  🔄 Found checkpoint: {ckpt_path}. Resuming...")
        start_epoch_actual, metrics = load_checkpoint(ckpt_path, model, optimizer, scheduler)
        start_epoch = start_epoch_actual + 1
        
        if start_epoch < config["warmup_epochs"]:
            if hasattr(model, 'module'):
                model.module.freeze_backbone()
            else:
                model.freeze_backbone()
                
        if "val_qwk" in metrics:
            best_qwk = metrics["val_qwk"]
            
        hist_path = os.path.join(model_dir, "training_history.csv")
        if os.path.exists(hist_path):
            try:
                hist_df = pd.read_csv(hist_path)
                history = hist_df.to_dict('list')
            except Exception as e:
                pass
        print(f"  ➡️ Resuming from Epoch {start_epoch + 1} (Best QWK: {best_qwk:.4f})")
    else:
        # --- Warmup: freeze backbone ---
        if hasattr(model, 'module'):
            model.module.freeze_backbone()
        else:
            model.freeze_backbone()

    for epoch in range(start_epoch, config["num_epochs"]):
        # Unfreeze after warmup
        if epoch == config["warmup_epochs"]:
            if hasattr(model, 'module'):
                model.module.unfreeze_backbone()
            else:
                model.unfreeze_backbone()
            # Reset optimizer for fine-tuning
            optimizer = optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)

        epoch_start = time.time()

        # Train
        use_mixup = epoch >= config["warmup_epochs"]
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, epoch,
            use_mixup=use_mixup, mixup_alpha=config["mixup_alpha"],
            scaler=scaler
        )

        # Validate
        val_loss, val_acc, val_qwk, val_f1, _, _, _ = validate(
            model, val_loader, criterion, device, use_amp=use_amp
        )

        # C2 fix: Pass epoch explicitly to CosineAnnealingWarmRestarts
        scheduler.step(epoch + 1)
        current_lr = optimizer.param_groups[0]["lr"]

        # Log history
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_qwk"].append(val_qwk)
        history["val_f1"].append(val_f1)
        history["lr"].append(current_lr)

        epoch_time = time.time() - epoch_start

        # Print progress
        print(f"  Epoch [{epoch+1:02d}/{config['num_epochs']}] "
              f"| loss: {train_loss:.4f}/{val_loss:.4f} "
              f"| acc: {train_acc:.4f}/{val_acc:.4f} "
              f"| QWK: {val_qwk:.4f} | F1: {val_f1:.4f} "
              f"| lr: {current_lr:.2e} | {epoch_time:.1f}s"
              + (" ⭐" if val_qwk > best_qwk else ""))

        # Save last checkpoint
        save_checkpoint(model, optimizer, scheduler, epoch, {
            "val_qwk": val_qwk, "val_acc": val_acc, "val_f1": val_f1
        }, os.path.join(model_dir, "last_model.pth"), config)

        # Save best checkpoint
        if val_qwk > best_qwk:
            best_qwk = val_qwk
            patience_counter = 0
            save_checkpoint(model, optimizer, scheduler, epoch, {
                "val_qwk": val_qwk, "val_acc": val_acc, "val_f1": val_f1
            }, os.path.join(model_dir, "best_model.pth"), config)
        else:
            patience_counter += 1

        # W4: Early stopping
        if patience_counter >= 7:
            print(f"  ⏹️ Early stopping at epoch {epoch+1} (no improvement for 7 epochs)")
            break

        # Fix for Kaggle Memory Leak: Force explicit garbage collection at epoch end
        gc.collect()
        torch.cuda.empty_cache()

    total_time = time.time() - start_time
    print(f"\n  ✅ {model_name} training complete in {total_time/60:.1f} min")
    print(f"  📊 Best val QWK: {best_qwk:.4f}")

    # Save history
    pd.DataFrame(history).to_csv(
        os.path.join(model_dir, "training_history.csv"), index=False
    )

    # Cleanup
    del model
    gc.collect()
    torch.cuda.empty_cache()

    return history, best_qwk

print("✅ Training pipeline defined")

✅ Training pipeline defined


In [ ]:
# ============================================================
# Train EfficientNet-B3
# ============================================================
print("\n" + "🚀" * 30)
print("  TRAINING EFFICIENTNET-B3")
print("🚀" * 30)

history_b3, best_qwk_b3 = train_model(
    "efficientnet_b3", train_loader, val_loader, class_weights, CONFIG, DEVICE
)


🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀
  TRAINING EFFICIENTNET-B3
🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀🚀

  Training: efficientnet_b3


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

  Using 2 GPUs with DataParallel
  🔒 Backbone frozen


  Epoch [01/30] | loss: 1.7041/1.5092 | acc: 0.2256/0.3012 | QWK: 0.3638 | F1: 0.2402 | lr: 9.76e-05 | 305.9s ⭐


  Epoch [02/30] | loss: 1.5962/1.4158 | acc: 0.2652/0.3735 | QWK: 0.5049 | F1: 0.3118 | lr: 9.05e-05 | 301.3s ⭐


  Epoch [03/30] | loss: 1.6219/1.3826 | acc: 0.2774/0.3735 | QWK: 0.5636 | F1: 0.3809 | lr: 7.94e-05 | 306.5s ⭐
  🔓 Backbone unfrozen


  Epoch [04/30] | loss: 1.5216/1.3165 | acc: 0.2988/0.4819 | QWK: 0.6567 | F1: 0.3893 | lr: 6.55e-05 | 319.1s ⭐


  Epoch [05/30] | loss: 1.4669/1.3734 | acc: 0.3079/0.4337 | QWK: 0.6098 | F1: 0.3587 | lr: 5.01e-05 | 314.4s


Epoch 6 Train:  15%|█▍        | 6/41 [00:40<04:08,  7.11s/it, loss=1.1209]

In [ ]:
# ============================================================
# Train EfficientNet-B4
# ============================================================
print("\n" + "🚀" * 30)
print("  TRAINING EFFICIENTNET-B4")
print("🚀" * 30)

# B4 has a larger memory footprint than B3, so build loaders with a safer batch size.
config_b4 = copy.deepcopy(CONFIG)
config_b4["batch_size"] = config_b4.get("batch_size_b4", max(1, config_b4["batch_size"] // 2))

train_loader_b4, val_loader_b4, _, class_weights_b4, _, _ = create_dataloaders(
    train_df, test_df, config_b4
)

history_b4, best_qwk_b4 = train_model(
    "efficientnet_b4", train_loader_b4, val_loader_b4, class_weights_b4, config_b4, DEVICE
)

## 🔮 Section 12: Ensemble Inference

In [ ]:
# ============================================================
# Load best models and run ensemble inference on test set
# ============================================================
@torch.no_grad()
def ensemble_predict(model_names, test_loader, config, device):
    all_probs = []

    for model_name in model_names:
        print(f"  Loading best checkpoint for {model_name}...")
        model = DRClassifier(
            model_name=model_name,
            num_classes=config["num_classes"],
            pretrained=False,
            drop_rate=config["drop_rate"],
        ).to(device)

        ckpt_path = os.path.join(config["output_dir"], model_name, "best_model.pth")
        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()

        model_probs = []
        model_probs_tta = []

        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)

            # Original prediction
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            model_probs.append(probs.cpu().numpy())

            # TTA: horizontal flip
            outputs_flip = model(torch.flip(images, dims=[3]))
            probs_flip = torch.softmax(outputs_flip, dim=1)
            model_probs_tta.append(probs_flip.cpu().numpy())

        # Average original + TTA
        model_probs = np.concatenate(model_probs, axis=0)
        model_probs_tta = np.concatenate(model_probs_tta, axis=0)
        avg_probs = (model_probs + model_probs_tta) / 2.0
        all_probs.append(avg_probs)

        ckpt_metrics = checkpoint.get("metrics", {})
        print(f"    Best val QWK: {ckpt_metrics.get('val_qwk', 'N/A')}")

        del model
        gc.collect()
        torch.cuda.empty_cache()

    # Ensemble: average probabilities
    ensemble_probs = np.mean(all_probs, axis=0)
    ensemble_preds = np.argmax(ensemble_probs, axis=1)

    # Get true labels
    true_labels = []
    for _, labels in test_loader:
        true_labels.extend(labels.numpy())
    true_labels = np.array(true_labels)

    return ensemble_preds, true_labels, ensemble_probs, all_probs

ensemble_preds, true_labels, ensemble_probs, individual_probs = ensemble_predict(
    CONFIG["model_names"], test_loader, CONFIG, DEVICE
)

# Quick metrics
ens_acc = accuracy_score(true_labels, ensemble_preds)
ens_qwk = cohen_kappa_score(true_labels, ensemble_preds, weights='quadratic')
ens_f1 = f1_score(true_labels, ensemble_preds, average='macro')

print(f"\n{'='*50}")
print(f"  🏆 ENSEMBLE TEST RESULTS")
print(f"{'='*50}")
print(f"  Accuracy:  {ens_acc:.4f} ({ens_acc*100:.2f}%)")
print(f"  QWK:       {ens_qwk:.4f}")
print(f"  Macro F1:  {ens_f1:.4f}")

# Individual model results
for i, name in enumerate(CONFIG["model_names"]):
    ind_preds = np.argmax(individual_probs[i], axis=1)
    ind_acc = accuracy_score(true_labels, ind_preds)
    ind_qwk = cohen_kappa_score(true_labels, ind_preds, weights='quadratic')
    print(f"  {name}: Acc={ind_acc:.4f}, QWK={ind_qwk:.4f}")

## 📋 Section 13: Classification Report

In [ ]:
# ============================================================
# Detailed Classification Report
# ============================================================
report_str = classification_report(
    true_labels, ensemble_preds,
    target_names=CONFIG["class_names"],
    digits=4
)
print("\n" + "="*70)
print("  CLASSIFICATION REPORT — Ensemble (EfficientNet-B3 + B4 + TTA)")
print("="*70)
print(report_str)

# Save to file
report_path = os.path.join(CONFIG["output_dir"], "ensemble", "classification_report.txt")
with open(report_path, "w") as f:
    f.write(f"CLASSIFICATION REPORT — Ensemble (EfficientNet-B3 + B4 + TTA)\n")
    f.write(f"{'='*70}\n")
    f.write(report_str)
    f.write(f"\nOverall Accuracy: {ens_acc:.4f}\n")
    f.write(f"Quadratic Weighted Kappa: {ens_qwk:.4f}\n")
    f.write(f"Macro F1-Score: {ens_f1:.4f}\n")
print(f"✅ Report saved to: {report_path}")

## 🎯 Section 14: Confusion Matrix

In [ ]:
# ============================================================
# Confusion Matrix Visualization
# ============================================================
cm = confusion_matrix(true_labels, ensemble_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=[f'G{i}' for i in range(5)],
            yticklabels=[f'G{i}' for i in range(5)],
            cbar_kws={'label': 'Count'},
            linewidths=0.5, linecolor='white',
            annot_kws={'fontsize': 14, 'fontweight': 'bold'})
axes[0].set_xlabel('Predicted Grade', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Grade', fontsize=12, fontweight='bold')
axes[0].set_title('Confusion Matrix (Raw Counts)', fontsize=13, fontweight='bold')

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='YlOrRd', ax=axes[1],
            xticklabels=[f'G{i}' for i in range(5)],
            yticklabels=[f'G{i}' for i in range(5)],
            cbar_kws={'label': 'Rate'},
            linewidths=0.5, linecolor='white',
            vmin=0, vmax=1,
            annot_kws={'fontsize': 13, 'fontweight': 'bold'})
axes[1].set_xlabel('Predicted Grade', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Grade', fontsize=12, fontweight='bold')
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')

plt.suptitle(f'Ensemble Results — Acc: {ens_acc:.2%} | QWK: {ens_qwk:.4f} | F1: {ens_f1:.4f}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

cm_path = os.path.join(CONFIG["output_dir"], "ensemble", "confusion_matrix.png")
plt.savefig(cm_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"✅ Confusion matrix saved to: {cm_path}")

## 📈 Section 15: Training Curves

In [ ]:
# ============================================================
# Training Metrics Visualization
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

model_colors = {'efficientnet_b3': ('#3498db', '#85c1e9'), 'efficientnet_b4': ('#e74c3c', '#f1948a')}

for model_name in CONFIG["model_names"]:
    hist_path = os.path.join(CONFIG["output_dir"], model_name, "training_history.csv")
    h = pd.read_csv(hist_path)
    c1, c2 = model_colors[model_name]
    label = model_name.replace('efficientnet_', 'EfficientNet-').upper()

    # Loss
    axes[0, 0].plot(h["epoch"], h["train_loss"], color=c1, linestyle='-', label=f'{label} Train', linewidth=2)
    axes[0, 0].plot(h["epoch"], h["val_loss"], color=c2, linestyle='--', label=f'{label} Val', linewidth=2)

    # Accuracy
    axes[0, 1].plot(h["epoch"], h["train_acc"], color=c1, linestyle='-', label=f'{label} Train', linewidth=2)
    axes[0, 1].plot(h["epoch"], h["val_acc"], color=c2, linestyle='--', label=f'{label} Val', linewidth=2)

    # QWK
    axes[1, 0].plot(h["epoch"], h["val_qwk"], color=c1, linestyle='-', marker='o',
                     markersize=3, label=f'{label}', linewidth=2)

    # LR
    axes[1, 1].plot(h["epoch"], h["lr"], color=c1, linestyle='-', label=f'{label}', linewidth=2)

titles = ['Loss (Train vs Val)', 'Accuracy (Train vs Val)',
          'Validation QWK (Quadratic Weighted Kappa)', 'Learning Rate Schedule']
ylabels = ['Loss', 'Accuracy', 'QWK', 'Learning Rate']

for ax, title, ylabel in zip(axes.flatten(), titles, ylabels):
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.legend(fontsize=9, loc='best')
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Training Metrics — EfficientNet-B3 vs B4', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()

curves_path = os.path.join(CONFIG["output_dir"], "ensemble", "training_curves.png")
plt.savefig(curves_path, dpi=200, bbox_inches='tight')
plt.show()
print(f"✅ Training curves saved to: {curves_path}")

## 📝 Section 16: Results Summary

In [ ]:
# ============================================================
# Final Results Summary
# ============================================================
print("\n" + "="*60)
print("  📊 FINAL RESULTS SUMMARY")
print("="*60)

# Per-model results
for i, name in enumerate(CONFIG["model_names"]):
    ind_preds = np.argmax(individual_probs[i], axis=1)
    ind_acc = accuracy_score(true_labels, ind_preds)
    ind_qwk = cohen_kappa_score(true_labels, ind_preds, weights='quadratic')
    ind_f1 = f1_score(true_labels, ind_preds, average='macro')
    label = name.replace('efficientnet_', 'EfficientNet-').upper()
    print(f"  {label}:")
    print(f"    Accuracy:  {ind_acc:.4f} ({ind_acc*100:.2f}%)")
    print(f"    QWK:       {ind_qwk:.4f}")
    print(f"    Macro F1:  {ind_f1:.4f}")
    print()

print(f"  🏆 ENSEMBLE (B3 + B4 + TTA):")
print(f"    Accuracy:  {ens_acc:.4f} ({ens_acc*100:.2f}%)")
print(f"    QWK:       {ens_qwk:.4f}")
print(f"    Macro F1:  {ens_f1:.4f}")
print()

# Output files
print("  📁 Output files:")
for root, dirs, files in os.walk(CONFIG["output_dir"]):
    for f in sorted(files):
        fpath = os.path.join(root, f)
        fsize = os.path.getsize(fpath)
        size_str = f"{fsize/1024:.1f}KB" if fsize < 1024*1024 else f"{fsize/1024/1024:.1f}MB"
        rel = os.path.relpath(fpath, CONFIG["output_dir"])
        print(f"    {rel} ({size_str})")

print("\n" + "="*60)
print("  ✅ ALL DONE! Notebook execution complete.")
print("="*60)